# LyricSlicer AI - Free GPU Testing via Google Colab

This notebook will spin up the LyricSlicer AI backend on a free Nvidia T4 GPU and use `ngrok` to create a secure public URL so you can test it from your local Windows machine.

## ⚠️ Prerequisites
1. Go to `Runtime > Change runtime type` in the menu above and ensure `Hardware accelerator` is set to **T4 GPU**.
2. You must push your latest local code to a public GitHub repository so Colab can clone it.
3. Get a free Auth Token from [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken) and paste it below.

In [ ]:
# 1. Setup Your Variables Here
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/LyricSlicerAI.git" # <-- Change this!
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN" # <-- Paste your ngrok token here!

In [ ]:
# 2. Clone the Repository
import os
!git clone {GITHUB_REPO_URL}
os.chdir('LyricSlicerAI/Backend')

In [ ]:
# 3. Install Dependencies
!apt-get update && apt-get install -y ffmpeg
!pip install fastapi uvicorn python-multipart pyngrok requests
!pip install git+https://github.com/m-bain/whisperx.git

In [ ]:
# 4. Setup Ngrok and Start Server
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import threading

# Authenticate ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open a public tunnel to port 8000
public_url = ngrok.connect(8000).public_url
print("\n" + "="*50)
print(f"🚀 PUBLIC API URL: {public_url}")
print("Copy this URL and use it with your local test_client.py!")
print("="*50 + "\n")

# Start the FastAPI server in the background
from main import app
nest_asyncio.apply()

# Run Uvicorn in a separate thread so it doesn't block the notebook
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()
